In [1]:
%load_ext autoreload
%autoreload 2

from Utils import Notebook
from Utils import Tex
from IPython.display import Math, display
import numpy as np
import scipy.linalg as la

from IPython.display import display, Math, Latex,Markdown

import ControllerDesigner


Notebook.setup()

LaTeX has been enabled for text rendering.


### Definição da Planta

In [2]:
def check_system_properties(A, B, C, tol=1e-9):
  n = A.shape[0]
  controllability = B.copy()
  for i in range(1, n):
    controllability = np.hstack(
        (controllability, np.linalg.matrix_power(A, i) @ B))
  rank_ctrb = np.linalg.matrix_rank(controllability, tol=tol)
  controllable = rank_ctrb == n

  observability = C.copy()
  for i in range(1, n):
    observability = np.vstack(
        (observability, C @ np.linalg.matrix_power(A, i)))
  rank_obsv = np.linalg.matrix_rank(observability, tol=tol)
  observable = rank_obsv == n

  eig_A = np.linalg.eigvals(A)
  stabilizable = True

  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.hstack((eig * np.eye(n) - A, B))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        stabilizable = False
        break

  detectable = True
  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.vstack((eig * np.eye(n) - A, C))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        detectable = False
        break

  return {
      "controllability_rank": rank_ctrb,
      "observability_rank": rank_obsv,
      "controllable": controllable,
      "observable": observable,
      "stabilizable": stabilizable,
      "detectable": detectable,
      "eigenvalues": eig_A,
  }


A = np.array([
    [0.0, 1.0],
    [-4.0, 0.4]
])

B = np.array([
    [0.0],
    [1.0]
])

C = np.array([
    [1.0, 0.0]
])

results = check_system_properties(A, B, C)

print("=" * 60)
print("SYSTEM PROPERTIES")
print("=" * 60)

print(f"Controllability rank : "
      f"{results['controllability_rank']} / {A.shape[0]}")

print(f"Observability rank   : "
      f"{results['observability_rank']} / {A.shape[0]}")

print(f"Controllable         : {results['controllable']}")
print(f"Observable           : {results['observable']}")
print(f"Stabilizable         : {results['stabilizable']}")
print(f"Detectable           : {results['detectable']}")

print("\nEigenvalues of A:")
for eig in results["eigenvalues"]:
  print(f"  {eig}")

SYSTEM PROPERTIES
Controllability rank : 2 / 2
Observability rank   : 2 / 2
Controllable         : True
Observable           : True
Stabilizable         : True
Detectable           : True

Eigenvalues of A:
  (0.20000000000000012+1.98997487421324j)
  (0.20000000000000012-1.98997487421324j)


### Co-projeto do Controlador baseado em Eventos

In [6]:
h = 1e-2
lambd = 1e-2
upsilon = 1e-3


# A = np.array([
#     [0.0, 1.0],
#     [0.0, -0.1]
# ])

# B = np.array([
#     [0.0],
#     [-0.1]
# ])

# C = np.array([
#     [1.0, 0.0],
#     # [4.0, 0.1]
# ])

A = np.array([
    [0.0, 1.0],
    [-4.0, 0.4]
])

B = np.array([
    [0.0],
    [1.0]
])

C = np.array([
    [0.0, 1.0],
    # [4.0, 0.1]
])


ctrl_params = {'A': A, 'B': B, 'C': C, 'h': h, 'λ': lambd,
               'υ1': upsilon, 'υ2': upsilon}
synth_res = ControllerDesigner.synthesize_output_based_setm(
    ctrl_params, eps=1e-6)

if synth_res is None:
  print("ERRO: Síntese Infeasible ou falha na recuperação das matrizes.")
else:
  Xi = synth_res['etm']['Ξ']
  Psi = synth_res['etm']['Ψ']
  K = synth_res['controller']['K']
  P = synth_res['functional']['P']
  cost = synth_res['optimal_value']

  print("\n=== GANHO E MATRIZES DE ACIONAMENTO ===")
  display(Math(rf'''
        K = {Tex.mat2tex(K)}, \\[8pt] P = {Tex.mat2tex(P)} \\[8pt]
        \Xi = {Tex.mat2tex(Xi)}, \\[8pt] \Psi = {Tex.mat2tex(Psi)}
    '''))


=== GANHO E MATRIZES DE ACIONAMENTO ===


<IPython.core.display.Math object>

### Projeto do Observador Impulsivo

In [9]:
from scipy.linalg import expm

obsr_params_discrete = {
    **ctrl_params,
    "lambda_obs": 1.0,  # Taxa mínima de decaimento por modo
    "nu_bar": 0.20       # nu_max em segundos
}

res_discrete = ControllerDesigner.synthesize_impulsive_observer_discrete_exact(
    obsr_params_discrete,
    eps=1e-6,
    verbose=False
)

if res_discrete is not None:
  L_disc = res_discrete["L"]
  h_step = obsr_params_discrete["h"]
  n_modes = res_discrete["nu_bar_steps"]

  display(Markdown(
      rf"**Síntese Discreta Exata:** Factível ($\gamma_L = {res_discrete['gamma_L']:.4e}$)"))
  display(Markdown(
      f"* Modos testados: $\\ell = 1, \\dots, {n_modes}$ ($\\tau \\in [{h_step:.2f}, {n_modes*h_step:.2f}]$ s)"))
  print("Ganho L reconstruído:\n", np.round(L_disc, 4))

  print("\nVerificação de estabilidade modal (Raio Espectral < 1):")
  for l in range(1, n_modes + 1):
    tau_l = l * h_step
    Phi_l = expm(ctrl_params["A"] * tau_l)
    A_err_mode = (
        np.eye(ctrl_params["A"].shape[0]) - L_disc @ ctrl_params["C"]) @ Phi_l
    rho_mode = np.max(np.abs(np.linalg.eigvals(A_err_mode)))
    print(
        f"Modo l={l:2d} (tau={tau_l:.2f}s) -> Raio Espectral = {rho_mode:.4f}")
else:
  display(Markdown(
      "**Síntese Discreta:** Infactível para a taxa de decaimento informada."))

**Síntese Discreta Exata:** Factível ($\gamma_L = 2.3970e-05$)

* Modos testados: $\ell = 1, \dots, 20$ ($\tau \in [0.01, 0.20]$ s)

Ganho L reconstruído:
 [[-0.2768]
 [ 0.5718]]

Verificação de estabilidade modal (Raio Espectral < 1):
Modo l= 1 (tau=0.01s) -> Raio Espectral = 0.9797
Modo l= 2 (tau=0.02s) -> Raio Espectral = 0.9574
Modo l= 3 (tau=0.03s) -> Raio Espectral = 0.9328
Modo l= 4 (tau=0.04s) -> Raio Espectral = 0.9053
Modo l= 5 (tau=0.05s) -> Raio Espectral = 0.8740
Modo l= 6 (tau=0.06s) -> Raio Espectral = 0.8374
Modo l= 7 (tau=0.07s) -> Raio Espectral = 0.7915
Modo l= 8 (tau=0.08s) -> Raio Espectral = 0.7196
Modo l= 9 (tau=0.09s) -> Raio Espectral = 0.6663
Modo l=10 (tau=0.10s) -> Raio Espectral = 0.6676
Modo l=11 (tau=0.11s) -> Raio Espectral = 0.6689
Modo l=12 (tau=0.12s) -> Raio Espectral = 0.6703
Modo l=13 (tau=0.13s) -> Raio Espectral = 0.6716
Modo l=14 (tau=0.14s) -> Raio Espectral = 0.6730
Modo l=15 (tau=0.15s) -> Raio Espectral = 0.6743
Modo l=16 (tau=0.16s) -> Raio Espectral = 0.6757
Modo l=17 (tau=0.17s) -> Raio Espectral = 0.6770
Modo l=18 (tau=0.18s) -> Raio Espectral = 0.6784
Modo l=19 (tau=

In [10]:
import numpy as np


def format_matrix_to_cpp(mat, name="X", scientific=True):
  """
  Converte um vetor ou matriz numpy para o formato C++:
  X = {{..., ...}, {..., ...}}
  """
  mat = np.atleast_2d(mat)
  rows, cols = mat.shape

  fmt = "{:.2e}" if scientific else "{:.4f}"

  rows_str = []
  for i in range(rows):
    row_vals = ", ".join([fmt.format(val) for val in mat[i]])
    rows_str.append(f"{row_vals}")

  inner_str = ", ".join(rows_str)
  return f"{name} = {{{inner_str}}};"


# Extração das variáveis do seu escopo
Xi = synth_res['etm']['Ξ']
Psi = synth_res['etm']['Ψ']
K = synth_res['controller']['K']
L = res_discrete["L"]

# Exibição no console / Jupyter no formato exato solicitado
print(format_matrix_to_cpp(Xi, name="Xi"))
print(format_matrix_to_cpp(Psi, name="Psi"))
print(format_matrix_to_cpp(K, name="K"))
print(format_matrix_to_cpp(L, name="L"))

Xi = {7.98e+00};
Psi = {6.23e-01};
K = {1.93e+00, -2.59e+00};
L = {-2.77e-01, 5.72e-01};
